# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Đọc các split do prepare_data.py tạo; không chia ngẫu nhiên lại.
import json
import random
from collections import Counter
from pathlib import Path

from PIL import Image, ImageDraw
from IPython.display import display
from scripts.labcommon import CLASSES, read_jsonl

DATA_DIR = Path("data")
splits = {
    name: read_jsonl(DATA_DIR / "splits" / f"{name}.jsonl")
    for name in ("train", "val", "test")
}
train_samples = splits["train"]

# 1. Thống kê số ảnh, số box từng lớp và số phiên quay.
for name, samples in splits.items():
    counts = Counter(obj["label"] for sample in samples for obj in sample["objects"])
    sessions = {sample["session"] for sample in samples}
    print(f"{name:5s}: {len(samples):4d} ảnh | {sum(counts.values()):4d} box "
          f"| {len(sessions)} session | "
          + ", ".join(f"{label}={counts[label]}" for label in CLASSES))

# 2. Mỗi session phải thuộc duy nhất một split.
session_sets = {
    name: {sample["session"] for sample in samples}
    for name, samples in splits.items()
}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = session_sets[a] & session_sets[b]
    assert not overlap, f"Trùng session giữa {a}/{b}: {sorted(overlap)}"
print("OK: train/val/test không trùng session.")

# 3. Xem cấu trúc một mẫu: đường dẫn ảnh, session và nhãn YOLO.
assert train_samples, "Split train rỗng. Kiểm tra bước chuẩn bị dữ liệu."
print(json.dumps(train_samples[0], ensure_ascii=False, indent=2))

# 4. YOLO [xc, yc, w, h] chuẩn hoá [0,1] -> box pixel để vẽ.
COLORS = {"pothole": "red", "crack": "orange", "manhole": "cyan"}

def draw_sample(sample):
    with Image.open(DATA_DIR / sample["image_path"]) as source:
        image = source.convert("RGB")
    width, height = image.size
    draw = ImageDraw.Draw(image)
    for obj in sample["objects"]:
        xc, yc, w, h = obj["yolo"]
        x1 = max(0, min(width - 1, (xc - w / 2) * width))
        y1 = max(0, min(height - 1, (yc - h / 2) * height))
        x2 = max(0, min(width - 1, (xc + w / 2) * width))
        y2 = max(0, min(height - 1, (yc + h / 2) * height))
        color = COLORS[obj["label"]]
        draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
        draw.text((x1, max(0, y1 - 12)), obj["label"], fill=color,
                  stroke_width=1, stroke_fill="black")
    return image

# Seed cố định chỉ để chọn mẫu minh hoạ, không thay đổi split.
examples = random.Random(42).sample(train_samples, min(6, len(train_samples)))
print("Màu box: pothole=đỏ, crack=cam, manhole=xanh cyan")
for sample in examples:
    print(f"{sample['image_id']} | session={sample['session']} "
          f"| {len(sample['objects'])} box")
    display(draw_sample(sample))


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9